# 第一章：单细胞RNA测序技术原理

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

单细胞 RNA-seq（scRNA-seq）已经从"新技术"变成了生命科学研究的基础设施。2019 年至今，PubMed 上每年新增的单细胞相关论文已超过万篇。但如果你第一次面对一个真实的 scRNA-seq 数据集，从 GEO 下载到最终的生物学结论，中间有太多"教程不会告诉你"的坑。

这个系列不是又一个"跑通代码"的教程。我们会用一个真实的疾病数据集——人类肝硬化单细胞图谱，走完从原始数据到高级分析的完整流程。每一步不仅讲 HOW，更讲 WHY：为什么选这个参数、结果该怎么解读、和原始论文的结果一致吗。

读完全系列，你将掌握：

多样本 scRNA-seq 的标准处理流程（QC → 整合 → 注释 → 差异分析）
6 种高级分析方法（富集、通讯、TF 调控、组成、轨迹、共表达）
处理真实数据时必须知道的"踩坑"经验
如何从数据中讲出一个完整的生物学故事

先来看看最终结果——跟完 13 篇教程后，你将从原始 GEO 数据得到这样的全景图谱：

全景 UMAP：58,735 个细胞的细胞类型分布——每个点是一个细胞，颜色代表不同的细胞类型

当我们把健康和肝硬化的细胞拆开来看时，差异一目了然：

按条件拆分的 UMAP：左侧为健康肝脏，右侧为肝硬化——可以直观看到疾病对细胞组成的影响

## 什么是单细胞 RNA-seq？

### 从"平均值"到"每一个细胞"

在传统的 Bulk RNA-seq 中，我们把成千上万个细胞混在一起提取 RNA、建库、测序。得到的是一个"平均表达谱"——你知道某个基因在整个组织中的平均表达量，但你不知道这些表达来自哪些细胞。

打个比方：Bulk RNA-seq 就像把一杯水果沙冰搅碎后分析成分——你知道里面有草莓和蓝莓的成分，但你看不到每颗水果的样子。而 scRNA-seq 就像把每颗水果拿出来单独分析。

单细胞 RNA-seq（scRNA-seq）能对每一个细胞独立测序，得到每个细胞的转录组谱。这意味着你可以：

发现稀有细胞类型——即使只占总细胞数的 1%，也能被检测到
追踪细胞状态变化——比如免疫细胞从静息态到激活态的连续过渡
揭示细胞间异质性——同一种"巨噬细胞"其实有多种亚型，功能完全不同
分析细胞间通讯——哪些细胞在"说话"，通过什么信号分子

### Bulk RNA-seq vs scRNA-seq 对比

对比维度
Bulk RNA-seq
scRNA-seq

分辨率
组织/群体水平
单细胞水平

细胞异质性
被平均掉，无法识别
可以捕获每个细胞的独特状态

稀有细胞
信号被稀释，难以检测
即使 <1% 也可发现

基因检测深度
深（每样本数千万 reads）
浅（每细胞仅数千 reads）

数据稀疏性
低（大多数基因可检出）
高（>90% 矩阵元素为 0，dropout 现象）

成本
低（~$200/样本）
高（~$2000-5000/样本）

样本需求
冻存/FFPE 组织均可
通常需要新鲜活细胞悬液

典型应用
组间差异表达、通路分析
细胞图谱、轨迹分析、通讯网络

简单来说：Bulk 告诉你"森林变了"，scRNA-seq 告诉你"森林里哪些树变了、怎么变的"。

💡 为什么 scRNA-seq 数据这么稀疏？

一个典型的人类细胞含有约 300,000 条 mRNA 分子，但 10x Chromium 平台只能捕获其中约 10-30%。加上逆转录效率的限制，最终每个细胞通常只检测到 1,000-5,000 个基因（人类基因组约有 20,000 个蛋白编码基因）。这意味着大量基因的表达信号被"丢失"了——这就是所谓的 dropout 现象，也是 scRNA-seq 分析中必须正视的技术限制。

但这并不是坏事——即使每个细胞只"看到"了 5,000 个基因，当你有 60,000 个细胞时，统计分析的力量足以弥补单个细胞的信息缺失。

## scRNA-seq 的技术原理：10x Genomics Chromium

目前最主流的 scRNA-seq 平台是 10x Genomics Chromium，它基于液滴微流控（droplet-based microfluidics）技术。本教程使用的数据（GSE136103）就是用这个平台产生的。全球超过 80% 的 scRNA-seq 论文使用了 10x 平台。

### 实验流程：从组织到测序

第 1 步：组织解离 → 单细胞悬液

将组织样本（如肝脏活检）用酶（胶原酶 + 透明质酸酶）消化，将细胞从组织基质中释放出来。这一步极其关键——消化不充分会丢失细胞，消化过度会损伤细胞膜。对于肝硬化这种富含纤维的组织，消化条件尤其需要优化。GSE136103 数据还对细胞悬液进行了 CD45 流式分选，将免疫细胞（CD45+）和非免疫细胞（CD45-）分开建库，以获得更均衡的细胞覆盖。

第 2 步：GEM 生成（液滴包裹）

单细胞悬液被注入 Chromium 芯片的微流控通道。在通道交汇处，每个细胞与一个 Gel Bead（凝胶微珠）一起被包裹在一个油滴中，形成 GEM（Gel Bead-in-Emulsion）。每个 Gel Bead 上附着了数百万条寡核苷酸引物，包含三个关键部分：

Barcode（细胞条码，16bp）——同一个 Gel Bead 上的所有引物共享同一个 barcode。这就是"知道每条 read 来自哪个细胞"的关键。10x v3 化学有约 600 万种不同的 barcode 序列。
UMI（Unique Molecular Identifier，12bp）——每条引物有一个独特的 UMI。用于消除 PCR 扩增偏差——同一个 RNA 分子被扩增 100 倍后，通过 UMI 我们知道它原本只有 1 个拷贝。
Poly(dT) 引物——捕获带有 poly(A) 尾巴的 mRNA。

第 3 步：细胞裂解 + 逆转录

在液滴内部，细胞膜被裂解剂破坏，mRNA 释放并被 Gel Bead 上的引物捕获。逆转录酶将 mRNA 转变为 cDNA，每条 cDNA 的 5' 端自动带上了 Barcode + UMI 标签。

第 4 步：cDNA 扩增 + 文库构建 + 测序

液滴破裂后，所有 cDNA 混合在一起进行 PCR 扩增、片段化、加接头，构建成 Illumina 文库并测序。测序后通过 Cell Ranger 管线将 reads 比对到参考基因组，通过 Barcode 分配回各个细胞，通过 UMI 去重，最终得到 Barcode × Gene 的计数矩阵——这就是我们分析的起点。

💡 Doublet 是怎么产生的？

如果两个细胞不幸被包裹在同一个液滴中，就会产生 doublet（双细胞）。10x 的 doublet rate 约为每加载 1000 个细胞增加 0.8%——加载 10,000 个细胞时约 8%。这些 doublet 会同时表达两种细胞类型的标记基因，在分析中制造虚假的"过渡态"。这就是为什么在后续的质量控制步骤中，我们需要用专门的算法（如 Scrublet）来检测和移除它们。

### 其他主流 scRNA-seq 技术

虽然本教程使用 10x Chromium 数据，但值得了解其他技术路线：

平台
原理
通量
特点

10x Chromium
液滴微流控
~10,000 细胞/样本
最主流，成熟度高

Drop-seq
液滴微流控
~5,000 细胞/样本
开源协议，成本较低

Smart-seq2/3
孔板分选
数百个细胞
全长转录本，灵敏度最高

sci-RNA-seq
组合索引
数十万细胞
超高通量，无需分离单细胞

## 生物学背景：肝脏——一个被低估的"复杂器官"

为什么选肝硬化数据集来做这个教程？因为肝脏是理解 scRNA-seq 价值的完美案例。

### 肝脏的细胞生态系统

肝脏是人体最大的实质性器官（成人约 1.2-1.5 kg），也是细胞类型最丰富的器官之一。它绝不仅仅是"一堆肝细胞"：

实质细胞（Parenchymal cells）

肝细胞（Hepatocytes）——约占肝脏体积的 80%，负责代谢、解毒、蛋白合成。肝细胞本身还存在 zonation（区带化）：靠近门静脉端（periportal）和靠近中央静脉端（pericentral）的肝细胞执行不同的代谢功能。但由于 scRNA-seq 需要组织消化，大体积的肝细胞容易在消化过程中破碎，因此在本数据集中肝细胞的比例偏低。
胆管细胞（Cholangiocytes）——组成胆管，负责修改和转运胆汁。在肝硬化中，可能出现胆管反应（ductular reaction），胆管细胞大量增生。

非实质细胞（Non-parenchymal cells）——虽然只占肝脏体积的 20%，却在疾病进程中扮演核心角色：

肝星状细胞（Hepatic Stellate Cells, HSCs）——正常情况下储存维生素 A，在肝损伤后被激活，转化为肌成纤维细胞（myofibroblasts），大量分泌胶原蛋白和纤维连接蛋白（fibronectin），是肝纤维化的直接执行者。这个过程是肝硬化的核心病理事件。
肝窦内皮细胞（Liver Sinusoidal Endothelial Cells, LSECs）——肝脏中独特的内皮细胞，带有 窗孔（fenestrae）——直径约 100nm 的小孔，允许血浆和小分子自由通过。在肝硬化中，LSECs 失去窗孔（去窗孔化，capillarization），不仅影响物质交换，还会释放促纤维化信号，进一步驱动 HSC 激活。
Kupffer 细胞——肝脏驻留巨噬细胞（tissue-resident macrophages），是人体最大的巨噬细胞群体。它们负责清除门静脉中的微生物和内毒素，维持肝脏的免疫耐受。在肝硬化中，Kupffer 细胞发生耗竭，取而代之的是从外周招募的 单核细胞来源的巨噬细胞——这两种巨噬细胞的功能截然不同。
T 细胞和 NK 细胞——肝脏是人体 NK 细胞最富集的器官之一（约占肝脏淋巴细胞的 30-50%）。在肝硬化中，T 细胞和 NK 细胞的组成和功能都发生显著变化——部分亚群被激活，另一些则出现耗竭（exhaustion）。

大类细胞注释 UMAP：本教程最终识别出的 12 种主要细胞类型，对应了上述的各种肝脏细胞

### 肝硬化：一场涉及所有细胞类型的"系统性重塑"

肝硬化是各种慢性肝病（乙肝、丙肝、酒精性肝病、非酒精性脂肪性肝病）的共同终末期表现。它不是一种单一的疾病，而是一个涉及所有肝脏细胞类型的级联反应：

起始损伤：病毒/酒精/脂肪持续损伤肝细胞，导致细胞死亡和炎症介质释放。
炎症放大：受损的肝细胞释放 DAMPs（damage-associated molecular patterns），激活 Kupffer 细胞和招募外周单核细胞。这些巨噬细胞分泌 TGF-β、PDGF 等促纤维化因子。
纤维化启动：HSCs 被 TGF-β 等信号激活，转化为肌成纤维细胞，大量沉积细胞外基质（ECM），形成纤维间隔（fibrous septa）。
血管重塑：LSECs 去窗孔化，新生血管形成，门静脉压力升高。
免疫微环境重编程：Kupffer 细胞耗竭、促纤维化巨噬细胞亚群扩增、T/NK 细胞功能障碍，形成"促纤维化-免疫抑制"的恶性循环。
终末期：正常肝脏结构被纤维组织和再生结节替代，肝功能衰竭。

传统的 Bulk RNA-seq 只能告诉你"整个肝脏的基因表达变了"——但到底是哪种细胞变了？是星状细胞还是巨噬细胞在驱动纤维化？不同的巨噬细胞亚群是促进还是抑制纤维化？这些问题只有 scRNA-seq 才能回答。

细胞类型比例变化：肝硬化中各细胞类型的比例变化——部分显著扩增（如 SAMac），部分显著减少（如 Kupffer 细胞）

### Ramachandran et al., 2019：改变肝脏研究的一篇论文

2019 年，Ramachandran 等人在 Nature 上发表了人类肝硬化的单细胞图谱（DOI: 10.1038/s41586-019-1631-3）。这篇论文不只是"又一个单细胞图谱"——它改变了我们对肝纤维化的理解：

发现 1：瘢痕相关巨噬细胞（SAMac）

论文最重要的发现是鉴定出一群全新的巨噬细胞亚群——TREM2+CD9+ 的瘢痕相关巨噬细胞（scar-associated macrophages, SAMac）。这群细胞几乎只存在于肝硬化组织的纤维化区域中，在健康肝脏中几乎检测不到。SAMac 高表达脂质代谢基因（如 TREM2、CD9、GPNMB）和促纤维化基因（如 SPP1），被认为是纤维化进程的关键驱动者之一。

发现 2：Kupffer 细胞的命运

论文发现肝硬化中 Kupffer 细胞（肝脏驻留巨噬细胞）被大量耗竭，取而代之的是来自外周血单核细胞分化而来的巨噬细胞。这意味着肝硬化不仅改变了巨噬细胞的"行为"，更替换了整个巨噬细胞的"队伍"。这两种来源的巨噬细胞具有完全不同的转录特征和功能——Kupffer 细胞偏向免疫耐受，而单核细胞来源的巨噬细胞偏向促炎和促纤维化。

发现 3：细胞通讯网络重构

通过配体-受体分析，论文揭示了 SAMac 与 HSC 之间的促纤维化通讯轴：SAMac 分泌 PDGFB 和 TNF 信号，驱动 HSC 持续激活和 ECM 沉积。这种细胞间通讯网络在健康肝脏中不存在——它是疾病状态下新建立的"病理通讯回路"。

配体-受体通讯分析：LIANA 分析揭示的细胞间通讯网络，展示不同细胞类型对之间最显著的信号通路

发现 4：内皮细胞亚群的重塑

内皮细胞不是铁板一块。论文鉴定出多种 LSEC 亚群，其中部分在肝硬化中显著扩增，伴随着窗孔相关基因（如 PECAM1、STAB2）的表达下降——这从单细胞层面证实了去窗孔化的发生。

💡 为什么不用 PBMC？

大多数 scRNA-seq 教程用 10x Genomics 官方的 PBMC 数据集（3,000 细胞、单样本）。那个数据太小、太简单——学完之后面对真实项目你还是不会做。真实项目的难点在于：多样本整合、batch effect、稀有细胞鉴定、疾病 vs 对照的差异分析。这些问题在 PBMC 教程里完全看不到。

GSE136103 有 10 个供体、20 个文库、~60,000 个细胞，包含健康和疾病两种条件——这才是你在实际科研中会遇到的数据规模和复杂度。

## 标准分析流程概览

拿到 Cell Ranger 输出的计数矩阵后，就进入了计算分析阶段。本系列 13 篇教程对应的完整流程如下：

### 第一阶段：数据处理（第 2-5 篇）

环境准备与数据加载（第 2 篇）——搭建分析环境，从 GEO 下载原始数据，读取 Cell Ranger 输出
质量控制（第 2 篇）——用 MAD 自适应过滤去除空液滴和损伤细胞，用 Scrublet 检测 doublet
归一化 + 降维 + 聚类（第 3 篇）——消除文库大小差异，选择高变基因，PCA 降维，UMAP 可视化，Leiden 聚类
批次矫正（第 4 篇）——用 Harmony 消除不同样本/文库间的技术差异
细胞注释（第 5 篇）——用标记基因和自动化工具给每个 cluster "取名字"

### 第二阶段：高级分析（第 6-13 篇）

差异表达（第 6 篇）——Wilcoxon + DESeq2 Pseudobulk，找到疾病改变了哪些基因
功能富集（第 7 篇）——GO/KEGG ORA + GSEA，从基因列表到生物学通路
细胞通讯（第 8 篇）——LIANA 配体-受体分析，揭示细胞间对话
转录因子网络（第 9 篇）——CollecTRI + decoupler，推断上游调控因子
细胞组成（第 10 篇）——比例检验和统计分析，量化疾病对细胞组成的影响
轨迹分析（第 11 篇）——PAGA + Diffusion Pseudotime，追踪细胞分化路径
共表达模块（第 12 篇）——基因集评分，发现协同表达的基因"套装"
高级可视化（第 13 篇）——制作出版级别的精美图表

完成这全部 13 篇教程后，你将得到包含 26 种细胞亚型的精细注释：

精细注释 UMAP：26 种细胞亚型——本系列的最终产出

## 本系列的结构

篇章
标题
对应分析步骤

01
单细胞技术原理与系列概览（本篇）
scRNA-seq 原理、生物学背景

02
质量控制
环境搭建、数据下载、QC 过滤、Doublet 检测

03
归一化、降维与聚类
LogNormalize、HVG、PCA、UMAP、Leiden

04
多样本整合
Harmony 批次矫正

05
多层细胞注释
Marker scoring + CellTypist + 亚群细分

06
差异表达分析
Wilcoxon + Pseudobulk DESeq2

07
功能富集分析
GO/KEGG ORA、GSEA preranked

08
细胞通讯分析
LIANA 配体-受体分析

09
转录因子调控网络
CollecTRI + decoupler ULM

10
细胞组成分析
比例检验、条件比较

11
轨迹与拟时序分析
PAGA、Diffusion Pseudotime

12
共表达与基因集评分
共表达模块、疾病通路评分

13
高级可视化与数据汇总
出版级图表、结果整合

每篇文章可以独立阅读，但建议按顺序跟完整流程。所有中间结果（.h5ad 文件）都可以从 GitHub 获取——如果你只对某个高级分析感兴趣，可以直接从第 5 篇之后开始。

## 小结

这一篇我们了解了：

✅ 什么是 scRNA-seq，以及它相比 Bulk RNA-seq 的革命性优势
✅ 10x Genomics Chromium 的技术原理：从组织消化到 Barcode × Gene 计数矩阵
✅ 肝脏的细胞生态系统：肝细胞、星状细胞、Kupffer 细胞、LSECs、T/NK 细胞
✅ 肝硬化的病理机制：从慢性损伤到纤维化的级联反应
✅ Ramachandran et al. 2019 的核心发现：SAMac、Kupffer 细胞耗竭、促纤维化通讯网络
✅ 本系列 13 篇教程的结构和分析流程概览

下一篇，我们将正式动手——搭建分析环境、从 GEO 下载 GSE136103 原始数据、读取 Cell Ranger v2 格式的计数矩阵，并对 81,448 个液滴进行严格的质量控制，筛选出 58,735 个高质量细胞。